In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

flight = pd.read_csv('data/flights.csv')

## Data preview and basic information
Each row represents a flight. The previews below retain the original data; negative delays indicate early departures or arrivals.

In [2]:
from IPython.display import display

# First 10 rows, with every column visible
with pd.option_context("display.max_columns", None):
    display(flight.head(10))

,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,name
0,0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,United Air Lines Inc.
1,1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00,United Air Lines Inc.
2,2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00,American Airlines Inc.
3,3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00,JetBlue Airways
4,4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01 06:00:00,Delta Air Lines Inc.
5,5,2013,1,1,554.0,558,-4.0,740.0,728,12.0,UA,1696,N39463,EWR,ORD,150.0,719,5,58,2013-01-01 05:00:00,United Air Lines Inc.
6,6,2013,1,1,555.0,600,-5.0,913.0,854,19.0,B6,507,N516JB,EWR,FLL,158.0,1065,6,0,2013-01-01 06:00:00,JetBlue Airways
7,7,2013,1,1,557.0,600,-3.0,709.0,723,-14.0,EV,5708,N829AS,LGA,IAD,53.0,229,6,0,2013-01-01 06:00:00,ExpressJet Airlines Inc.
8,8,2013,1,1,557.0,600,-3.0,838.0,846,-8.0,B6,79,N593JB,JFK,MCO,140.0,944,6,0,2013-01-01 06:00:00,JetBlue Airways
9,9,2013,1,1,558.0,600,-2.0,753.0,745,8.0,AA,301,N3ALAA,LGA,ORD,138.0,733,6,0,2013-01-01 06:00:00,American Airlines Inc.


In [3]:
# A reproducible sample from across the dataset
preview_columns = ["year", "month", "day", "carrier", "flight", "origin", "dest", "dep_delay", "arr_delay", "air_time", "distance"]
display(flight.sample(10, random_state=42)[preview_columns])

,year,month,day,carrier,flight,origin,dest,dep_delay,arr_delay,air_time,distance
159280,2013,3,25,UA,1471,EWR,RSW,24.0,19.0,169.0,1068
189101,2013,4,26,DL,1765,JFK,SFO,-4.0,-37.0,337.0,2586
212435,2013,5,21,EV,4129,EWR,DCA,11.0,16.0,39.0,199
266804,2013,7,18,EV,5796,EWR,CLT,-8.0,-22.0,77.0,529
306581,2013,8,29,B6,939,JFK,BQN,-5.0,0.0,198.0,1576
68828,2013,11,14,UA,314,JFK,LAX,6.0,2.0,342.0,2475
164026,2013,3,30,B6,675,JFK,LAX,0.0,-4.0,338.0,2475
289110,2013,8,10,DL,4,JFK,SJU,-1.0,2.0,205.0,1598
161286,2013,3,27,9E,4091,JFK,BWI,-5.0,-9.0,37.0,184
205307,2013,5,13,DL,245,JFK,PHX,-3.0,-59.0,262.0,2153


In [4]:
dates = pd.to_datetime(flight[["year", "month", "day"]])
print(f"Rows: {flight.shape[0]:,} | Columns: {flight.shape[1]}")
print(f"Date range: {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d}")
print(f"Origins: {flight.origin.nunique()} | Destinations: {flight.dest.nunique()} | Airlines: {flight.carrier.nunique()}")
print(f"Duplicate records excluding id: {flight.drop(columns="id").duplicated().sum():,}")
print()
flight.info()

Rows: 336,776 | Columns: 21
Date range: 2013-01-01 to 2013-12-31
Origins: 3 | Destinations: 105 | Airlines: 16


Duplicate records excluding id: 0



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336776 entries, 0 to 336775
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              336776 non-null  int64  
 1   year            336776 non-null  int64  
 2   month           336776 non-null  int64  
 3   day             336776 non-null  int64  
 4   dep_time        328521 non-null  float64
 5   sched_dep_time  336776 non-null  int64  
 6   dep_delay       328521 non-null  float64
 7   arr_time        328063 non-null  float64
 8   sched_arr_time  336776 non-null  int64  
 9   arr_delay       327346 non-null  float64
 10  carrier         336776 non-null  object 
 11  flight          336776 non-null  int64  
 12  tailnum         334264 non-null  object 
 13  origin          336776 non-null  object 
 14  dest            336776 non-null  object 
 15  air_time        327346 non-null  float64
 16  distance        336776 non-null  int64  
 17  hour      

In [4]:
# Data types, missing values, and number of distinct values
column_summary = pd.DataFrame({
    "dtype": flight.dtypes.astype(str),
    "non_null": flight.notna().sum(),
    "missing": flight.isna().sum(),
    "missing_pct": flight.isna().mean().mul(100).round(2),
    "unique_values": flight.nunique()
})
display(column_summary)

,dtype,non_null,missing,missing_pct,unique_values
id,int64,336776,0,0.00,336776
year,int64,336776,0,0.00,1
month,int64,336776,0,0.00,12
day,int64,336776,0,0.00,31
dep_time,float64,328521,8255,2.45,1318
sched_dep_time,int64,336776,0,0.00,1021
dep_delay,float64,328521,8255,2.45,527
arr_time,float64,328063,8713,2.59,1411
sched_arr_time,int64,336776,0,0.00,1163
arr_delay,float64,327346,9430,2.80,577


In [5]:
# Summary statistics (missing values excluded)
display(flight[["dep_delay", "arr_delay", "air_time", "distance"]].describe().round(2))

,dep_delay,arr_delay,air_time,distance
count,328521.00,327346.00,327346.00,336776.00
mean,12.64,6.90,150.69,1039.91
std,40.21,44.63,93.69,733.23
min,-43.00,-86.00,20.00,17.00
25%,-5.00,-17.00,82.00,502.00
50%,-2.00,-5.00,129.00,872.00
75%,11.00,14.00,192.00,1389.00
max,1301.00,1272.00,695.00,4983.00


### Initial observations
- `origin` and `dest` are categorical location features suitable for the encoding comparison.
- `time_hour` loads as text and should be parsed as a datetime when needed.
- `id` is an identifier, rather than a meaningful numeric predictor.
- Delay and air-time fields contain missing values; account for these before modeling.
- The delay means exceed the medians, indicating that large delays pull the averages upward.

## Data cleaning issues and handling

The source dataset contains **336,776 flights and 21 columns**. The following checks were performed on `data/flights.csv` before documenting the encoding workflow.

### Missing values

| Column | Missing rows | Percentage of all flights |
| --- | ---: | ---: |
| `dep_time` | 8,255 | 2.45% |
| `dep_delay` | 8,255 | 2.45% |
| `arr_time` | 8,713 | 2.59% |
| `arr_delay` | 9,430 | 2.80% |
| `tailnum` | 2,512 | 0.75% |
| `air_time` | 9,430 | 2.80% |

These counts overlap; they should not be added together. All other columns have no missing values.

**Handling already implemented:** `arr_delay` is the prediction target. The encoding code below uses `flight.dropna(subset=[TARGET_COLUMN])` before splitting the data. This excludes the **9,430 rows without an observed target**, leaving **327,346 flights** for the shared training/test split. In this dataset, those retained rows have **no missing values in any column**, so no additional missing-value imputation is needed for this subset. The original `flight` DataFrame remains unchanged. Missing arrival delays should not be replaced with zero, because zero means an observed on-time arrival. Results describe flights with recorded arrival delays; the reason each excluded value is missing has not been established from these checks.

### Time formats and predictor preparation

- `time_hour` loads as text. All values parse successfully as datetimes, but convert it with `pd.to_datetime` if using datetime features.
- `dep_time`, `sched_dep_time`, `arr_time`, and `sched_arr_time` use HHMM notation rather than elapsed minutes. There are **29 departure times and 150 arrival times recorded as `2400`**. Treat these as midnight and account for the date rollover when constructing timestamps; do not treat HHMM values as ordinary continuous measurements. No other invalid HHMM values were found among nonmissing entries.
- `id` is unique and serves as a row identifier. Exclude it from model predictors. The encoding code preserves it and the other non-airport columns; predictor selection and time conversion still need to be performed before fitting a model.

### Negative and extreme delays

Negative delays indicate early departures or arrivals and should be retained. Observed departure delays range from **-43 to 1,301 minutes**, and arrival delays range from **-86 to 1,272 minutes**. The large positive values warrant inspection for their influence on model performance, but their size alone does not establish that they are errors. The current code does not trim, cap, or remove these values.

### Checks that found no cleaning problems

- **No duplicate flight records** were found after excluding `id` from the comparison.
- `origin` and `dest` have **no missing values**, and every airport code is covered by the local coordinate lookup.
- No blank strings or leading/trailing whitespace were found in the loaded text columns.
- All year/month/day combinations and `time_hour` strings parse as valid dates or timestamps.
- Nonmissing `air_time` and `distance` values are positive.

Only the missing-target exclusion is applied by the existing encoding workflow. The preparation recommendations above are documented here without changing the encoding methods or the source data.


## Encoding `origin` and `dest`

We compare five representations of the two airport-code columns. All other columns
and original row indices are preserved, and `flight` is not modified.

| Strategy | Representation | Main tradeoff |
| --- | --- | --- |
| **One-hot** | One 0/1 column per airport, separately for origin and destination. | Preserves identity without order, but adds columns. |
| **Ordinal** | An integer ID for each airport, assigned alphabetically. | Compact, but the numeric order and distances are arbitrary. |
| **Frequency** | The airport's proportion of rows in the training set. | Compact; airports with equal frequency become indistinguishable. |
| **Smoothed target** | An airport's average outcome, shrunk toward the training mean. | Uses predictive information, but requires a target and careful fitting to avoid leakage. |
| **Geographic** | Latitude and longitude for each airport (four columns total). | Captures physical location, but cannot directly represent every airport-specific effect. |

The same 80/20 random split is used for every strategy. Category vocabularies and
frequencies are learned only from the training rows. Target encoding uses internal
five-fold cross-fitting for training rows; test rows use mappings learned from the
training set only. Coordinates come from a fixed external lookup, not outcomes.

This is a reusable holdout comparison, not a future-flight forecasting split. For
cross-validation, call `make_encoded_datasets` separately inside **each** fold on
the raw fold data; do not cross-validate precomputed target-encoded training data.

References: [scikit-learn TargetEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html),
[nycflights13 airport data](https://github.com/tidyverse/nycflights13/blob/main/data-raw/airports.csv),
and [airport-codes / OurAirports](https://github.com/datasets/airport-codes).

In [6]:
# Requires pandas, numpy, and scikit-learn >= 1.3 (TargetEncoder).
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder

AIRPORT_COLUMNS = ["origin", "dest"]
RANDOM_STATE = 42
TARGET_COLUMN = "arr_delay"  # Predict arrival delay in minutes.

# The lookup is included locally, so rerunning does not require a download.
airport_coordinates = pd.read_csv("data/airport_coordinates.csv").set_index("airport")
assert airport_coordinates.index.is_unique
assert airport_coordinates[["latitude", "longitude"]].notna().all().all()

### Reusable encoding function

Each result contains full `train` and `test` DataFrames with only the airport columns
replaced. Fitted encoders, frequency mappings, and generated column names are also
returned for inspection. One-hot columns use sparse storage to reduce memory use.

Unseen airports become all zeros for one-hot, `-1` for ordinal, `0` for frequency,
and the training target mean for target encoding. Geographic encoding raises an
error for missing coordinates instead of inventing a location.

For smoothed target encoding we use a prior weight of 20:
`(airport_count * airport_mean + 20 * global_mean) / (airport_count + 20)`.
Training values are computed without the row's own cross-fitting fold. This
implementation treats the target as a continuous outcome for regression.

In [7]:
def make_encoded_datasets(train, test, coordinates, target_column=None):
    """Fit on raw training rows and replace only origin/dest in both splits."""
    if not train.index.is_unique or not test.index.is_unique:
        raise ValueError("Row indices must be unique for reliable alignment.")
    if len(train.index.intersection(test.index)):
        raise ValueError("Training and test indices must be disjoint.")
    if not train.columns.equals(test.columns):
        raise ValueError("Training and test columns must match.")
    if train[AIRPORT_COLUMNS].isna().any().any() or test[AIRPORT_COLUMNS].isna().any().any():
        raise ValueError("Handle missing airport codes before encoding.")

    train_codes = train[AIRPORT_COLUMNS].astype(str)
    test_codes = test[AIRPORT_COLUMNS].astype(str)
    variants, fitted, feature_names = {}, {}, {}

    def save(name, train_features, test_features):
        # Joining by the original index preserves row/target alignment.
        variants[name] = {
            "train": pd.concat([train.drop(columns=AIRPORT_COLUMNS), train_features], axis=1),
            "test": pd.concat([test.drop(columns=AIRPORT_COLUMNS), test_features], axis=1),
        }
        feature_names[name] = train_features.columns.tolist()

    # 1. One-hot: one indicator for each airport observed in training.
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.int8)
    train_matrix = one_hot.fit_transform(train_codes)
    test_matrix = one_hot.transform(test_codes)
    names = one_hot.get_feature_names_out(AIRPORT_COLUMNS)
    save("one_hot",
         pd.DataFrame.sparse.from_spmatrix(train_matrix, index=train.index, columns=names),
         pd.DataFrame.sparse.from_spmatrix(test_matrix, index=test.index, columns=names))
    fitted["one_hot"] = one_hot

    # 2. Ordinal: arbitrary category IDs, not geographic rankings.
    ordinal = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                             dtype=np.int32)
    names = [f"{col}_ordinal" for col in AIRPORT_COLUMNS]
    save("ordinal",
         pd.DataFrame(ordinal.fit_transform(train_codes), index=train.index, columns=names),
         pd.DataFrame(ordinal.transform(test_codes), index=test.index, columns=names))
    fitted["ordinal"] = ordinal

    # 3. Frequency: use only the training distribution for both splits.
    frequency_maps = {col: train_codes[col].value_counts(normalize=True)
                      for col in AIRPORT_COLUMNS}
    def frequencies(codes):
        return pd.DataFrame({f"{col}_frequency": codes[col].map(frequency_maps[col]).fillna(0.0)
                             for col in AIRPORT_COLUMNS}, index=codes.index)
    save("frequency", frequencies(train_codes), frequencies(test_codes))
    fitted["frequency"] = frequency_maps

    # 4. Smoothed target: optional until the prediction target is selected.
    if target_column is not None:
        if target_column in AIRPORT_COLUMNS:
            raise ValueError("The target must be separate from origin and dest.")
        y_train = train[target_column]
        if not pd.api.types.is_numeric_dtype(y_train) or not np.isfinite(y_train).all():
            raise ValueError("Target encoding requires finite numeric training targets.")
        if len(train) < 5:
            raise ValueError("At least five training rows are needed for cross-fitting.")
        target_encoder = TargetEncoder(target_type="continuous", smooth=20.0,
                                       cv=5, shuffle=True, random_state=RANDOM_STATE)
        names = [f"{col}_target_mean" for col in AIRPORT_COLUMNS]
        # fit_transform is intentional: fit(...).transform(...) would leak targets.
        train_encoded = target_encoder.fit_transform(train_codes, y_train)
        test_encoded = target_encoder.transform(test_codes)
        save("smoothed_target",
             pd.DataFrame(train_encoded, index=train.index, columns=names),
             pd.DataFrame(test_encoded, index=test.index, columns=names))
        fitted["smoothed_target"] = target_encoder

    # 5. Geographic: fixed latitude/longitude lookup, independent of the target.
    def geographic(codes):
        features = pd.DataFrame(index=codes.index)
        for col in AIRPORT_COLUMNS:
            for coordinate in ["latitude", "longitude"]:
                features[f"{col}_{coordinate}"] = codes[col].map(coordinates[coordinate])
        if features.isna().any().any():
            missing = sorted(set(codes.to_numpy().ravel()) - set(coordinates.index))
            raise ValueError(f"Missing airport coordinates: {missing}")
        return features
    save("geographic", geographic(train_codes), geographic(test_codes))
    fitted["geographic"] = coordinates.copy()
    return variants, fitted, feature_names

### Create the modified datasets

Choose `TARGET_COLUMN` in the configuration cell when your prediction outcome is
decided. Until then, the four target-independent variants are created. Once a
target is set, rows missing that outcome are excluded **equally for all five
strategies**, then the shared split and encodings are rebuilt. Excluded rows remain
in the original `flight` DataFrame.

Access a dataset with `encoded_datasets["one_hot"]["train"]`, replacing `one_hot`
with `ordinal`, `frequency`, `smoothed_target`, or `geographic` as appropriate.
These are partially prepared datasets: other categorical columns, missing
predictors, identifiers, and features unavailable at prediction time still need
to be handled when building models. The target remains in each dataset and must
be removed from predictors before fitting a model.

In [8]:
if TARGET_COLUMN is None:
    encoding_data = flight.copy()
    print("Target not selected: creating four encodings; set TARGET_COLUMN for all five.")
else:
    if TARGET_COLUMN not in flight.columns or TARGET_COLUMN in AIRPORT_COLUMNS:
        raise ValueError("Choose an existing outcome column other than origin or dest.")
    encoding_data = flight.dropna(subset=[TARGET_COLUMN]).copy()
    print(f"Rows excluded for missing {TARGET_COLUMN}: {len(flight) - len(encoding_data):,}")

raw_train, raw_test = train_test_split(
    encoding_data, test_size=0.20, random_state=RANDOM_STATE
)
encoded_datasets, fitted_encoders, encoded_columns = make_encoded_datasets(
    raw_train, raw_test, airport_coordinates, target_column=TARGET_COLUMN
)

encoding_summary = pd.DataFrame([
    {"strategy": name,
     "train_rows": len(parts["train"]),
     "test_rows": len(parts["test"]),
     "airport_features": len(encoded_columns[name]),
     "total_columns": parts["train"].shape[1]}
    for name, parts in encoded_datasets.items()
]).set_index("strategy")
display(encoding_summary)

# Confirm the comparison changes only the two requested columns.
unchanged_columns = flight.columns.difference(AIRPORT_COLUMNS)
for name, parts in encoded_datasets.items():
    for split, original in [("train", raw_train), ("test", raw_test)]:
        result = parts[split]
        assert result.index.equals(original.index)
        pd.testing.assert_frame_equal(result[unchanged_columns], original[unchanged_columns])
        assert not set(AIRPORT_COLUMNS).intersection(result.columns)
        assert result[encoded_columns[name]].notna().all().all()
print("Verified: identical rows across strategies and all other columns preserved.")

Rows excluded for missing arr_delay: 9,430


C:\Users\ellie\anaconda3\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


,train_rows,test_rows,airport_features,total_columns
strategy,,,,
one_hot,261876,65470,107,126
ordinal,261876,65470,2,21
frequency,261876,65470,2,21
smoothed_target,261876,65470,2,21
geographic,261876,65470,4,23


Verified: identical rows across strategies and all other columns preserved.


In [9]:
# Preview just the newly encoded airport columns for the same five flights.
preview_index = raw_train.index[:5]
display(raw_train.loc[preview_index, AIRPORT_COLUMNS])
for strategy, parts in encoded_datasets.items():
    print(strategy)
    display(parts["train"].loc[preview_index, encoded_columns[strategy]])

# Later, reuse the same datasets for each model configuration, for example:
# train_one_hot = encoded_datasets["one_hot"]["train"]
# X_train = train_one_hot.drop(columns=[TARGET_COLUMN])
# y_train = train_one_hot[TARGET_COLUMN]
# Select/preprocess the remaining predictors before model fitting.

,origin,dest
17752,EWR,ORD
313100,JFK,DCA
126586,EWR,GSP
201277,EWR,DTW
102571,EWR,PBI


one_hot


,origin_EWR,origin_JFK,origin_LGA,dest_ABQ,dest_ACK,dest_ALB,dest_ANC,dest_ATL,dest_AUS,dest_AVL,...,dest_SNA,dest_SRQ,dest_STL,dest_STT,dest_SYR,dest_TPA,dest_TUL,dest_TVC,dest_TYS,dest_XNA
17752,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
313100,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
126586,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
201277,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
102571,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


ordinal


,origin_ordinal,dest_ordinal
17752,0,68
313100,1,28
126586,0,38
201277,0,32
102571,0,70


frequency


,origin_frequency,dest_frequency
17752,0.357906,0.050719
313100,0.333055,0.027784
126586,0.357906,0.002364
201277,0.357906,0.027418
102571,0.357906,0.019631


smoothed_target


,origin_target_mean,dest_target_mean
17752,9.112825,5.574670
313100,5.446162,8.939583
126586,9.033949,14.011700
201277,9.141325,5.806033
102571,9.027945,8.567636


geographic


,origin_latitude,origin_longitude,dest_latitude,dest_longitude
17752,40.692500,-74.168667,41.978603,-87.904842
313100,40.639751,-73.778925,38.852083,-77.037722
126586,40.692500,-74.168667,34.895556,-82.218889
201277,40.692500,-74.168667,42.212444,-83.353389
102571,40.692500,-74.168667,26.683161,-80.095589


In [26]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
train_one_hot = encoded_datasets["one_hot"]["train"]
X_train = train_one_hot.drop(columns=[TARGET_COLUMN, "arr_time", "sched_dep_time", "sched_arr_time"])
y_train = train_one_hot[TARGET_COLUMN]
X_train = X_train.select_dtypes(exclude=['category', 'object', 'string'])
test_one_hot = encoded_datasets["one_hot"]["test"]
X_test = test_one_hot.drop(columns=[TARGET_COLUMN, "arr_time", "sched_dep_time", "sched_arr_time"])
y_test = test_one_hot[TARGET_COLUMN]
X_test = X_test.select_dtypes(exclude=['category', 'object', 'string'])
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0, solver="lsqr", tol=1e-4, max_iter=10000, fit_intercept=True))
model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy:.2f}")
RFmodel = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=5, max_features=1.0, bootstrap=True, random_state=42, n_jobs=1)
RFmodel.fit(X_train, y_train)
RFaccuracy = RFmodel.score(X_test, y_test)
print(f"Random Forest Accuracy: {RFaccuracy:.2f}")
Gradientmodel = GradientBoostingRegressor(loss="squared_error", n_estimators=200, learning_rate=0.05, max_depth=3, min_samples_leaf=5, max_features=None, subsample=1.0, n_iter_no_change=None, random_state=42)
Gradientmodel.fit(X_train, y_train)
Gradientaccuracy = RFmodel.score(X_test, y_test)
print(f"Gradient Accuracy: {Gradientaccuracy:.2f}")

C:\Users\ellie\anaconda3\Lib\site-packages\sklearn\utils\validation.py:911: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(
C:\Users\ellie\anaconda3\Lib\site-packages\sklearn\utils\validation.py:911: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(
C:\Users\ellie\anaconda3\Lib\site-packages\sklearn\utils\validation.py:911: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(
C:\Users\ellie\anaconda3\Lib\site-packages\sklearn\utils\validation.py:911: UserWarning: pandas.DataFrame with sparse columns found.It will be converted to a dense numpy array.
  warnings.warn(


Accuracy: 0.89


KeyboardInterrupt: 